### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import json
import pandas as pd
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_processing.mongos_store import MongoStore
from quinex_application.quinex_processing.quinex_client import QuinexClient
from quinex_application.quinex_processing.quinex_postprocessing import QuinexPostprocessing
from quinex_application.postprocessing.spatial_normalization import SpatialNormalization

In [2]:
quinex = QuinexClient()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2024_closed")
qpost = QuinexPostprocessing()
spatial = SpatialNormalization()

### **Load Analysis from MongoDB**

In [ ]:
analysis = []
# "papers_2010_abstract_fallback"
# Jahre 2010–2025
for year in range(2010, 2026):
    analysis.append(f"papers_{year}")
    analysis.append(f"papers_{year}_closed")
    analysis.append(f"papers_{year}_abstract_fallback")

analysis += ["papers_wos", "papers_wos_closed", "papers_wos_abstract_fallback"]

meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_path = "../data/quinex/df_quinex_new.parquet"
df_metadata_path = "../data/quinex/df_quinex_metadata.parquet"

for ana in analysis:
    a = mongodb.get_collection(ana)
    print(ana, len(a))
    
    if "_abstract_fallback" not in ana:
        df_quinex_meta = qpost.create_metadata_df(a, meta_df)
        qpost.save_df(df_quinex_meta, df_metadata_path)

    df_quinex = qpost.create_information_df(a, meta_df)
    qpost.save_df(df_quinex, df_path)


### **Rerun Spatial Normalization**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
df = spatial.get_geo_information_from_spatial_scope(
    df_quinex, 
    nbr_runs_per_model=1, 
    models = [
        "7 - Qwen3-Coder-30B-A3B-Instruct - A code model from August 2025", 
        "0 - Ministral-3-14B-Instruct-2512 - The latest Ministral from Dec.2.2025",
        ]
    )

In [ ]:
df_path = "../data/quinex/df_quinex_refined_spatial_new2.parquet"
df["spatial_scope_refined"] = df["spatial_scope_refined"].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else None
).astype("string")

df.to_parquet(df_path, index=False)

In [ ]:
df = pd.read_parquet("../data/quinex/df_quinex_refined_spatial_new.parquet")
print(len(df))
df["publication_year"].value_counts()

### **Rerun Quantity Parsing**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
CHUNK_SIZE = 200_000
CHECKPOINT_DIR = "../data/quinex/checkpoints"
PARSE_TIMEOUT = 5  # seconds per row
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

total = len(df_quinex)
chunks_done = sorted([
    int(f.replace("chunk_", "").replace(".parquet", ""))
    for f in os.listdir(CHECKPOINT_DIR)
    if f.startswith("chunk_") and f.endswith(".parquet")
])

start_row = max(chunks_done) + CHUNK_SIZE if chunks_done else 0
print(f"Bereits abgeschlossene Chunks: {chunks_done}")
print(f"Starte bei Zeile {start_row} von {total}")

In [ ]:
for chunk_start in tqdm(range(start_row, total, CHUNK_SIZE), desc="Chunks"):
    chunk_end = min(chunk_start + CHUNK_SIZE, total)
    chunk = df_quinex.iloc[chunk_start:chunk_end].copy()

    # Process-basierter Timeout: hängende Parses werden nach PARSE_TIMEOUT Sekunden terminiert
    results = qpost.parse_batch_with_timeout(
        chunk["quantity_text"].tolist(),
        timeout=PARSE_TIMEOUT,
        desc=f"Rows {chunk_start}–{chunk_end}"
    )

    chunk["value_normalized_adapted"] = [r[0] for r in results]
    chunk["unit_normalized_adapted"] = [r[1] for r in results]

    # Serialisiere unit_normalized_adapted zu JSON-String (Dicts mit gemischten Typen sind nicht parquet-kompatibel)
    chunk["unit_normalized_adapted"] = chunk["unit_normalized_adapted"].apply(
        lambda x: json.dumps(x) if x is not None else None
    )
    chunk.to_parquet(f"{CHECKPOINT_DIR}/chunk_{chunk_start}.parquet", index=True)

In [ ]:
# Alle Chunks zusammenführen
print("Füge alle Chunks zusammen...")
all_chunk_files = sorted(
    [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("chunk_") and f.endswith(".parquet")],
    key=lambda f: int(f.replace("chunk_", "").replace(".parquet", ""))
)
df_parsed = pd.concat([pd.read_parquet(f"{CHECKPOINT_DIR}/{f}") for f in all_chunk_files])

# Deserialisiere unit_normalized_adapted zurück zu Listen von Dicts
df_parsed["unit_normalized_adapted"] = df_parsed["unit_normalized_adapted"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else None
)

df_quinex[["value_normalized_adapted", "unit_normalized_adapted"]] = df_parsed[["value_normalized_adapted", "unit_normalized_adapted"]]
print(f"Fertig! {len(df_quinex)} Zeilen verarbeitet.")

In [ ]:
df_quinex_save = df_quinex.copy()
df_quinex_save["unit_normalized_adapted"] = df_quinex_save["unit_normalized_adapted"].apply(
    lambda x: json.dumps(x) if x is not None else None
)
df_quinex_save.to_parquet("../data/quinex/df_quinex_adapted.parquet", index=False)

### **Rerun Unit Dimension**

In [3]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex_adapted.parquet")

In [4]:
df_quinex = qpost.apply_unit_dimension_parsing(df_quinex)

Parsing unit dimensions:   0%|          | 0/3244165 [00:00<?, ?it/s]

In [6]:
df_quinex.to_parquet("../data/quinex/df_quinex_adapted.parquet", index=False)

### **Load Analysis Metadata into DataFrame**

In [ ]:
meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")

df_quinex_meta = qpost.create_metadata_df(analysis, meta_df)

In [ ]:
df_path = "../data/quinex/df_quinex_metadata.parquet"
qpost.save_df(df_quinex_meta, df_path)

### **Load Analysis Results into DataFrame**

In [ ]:
meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")

df_quinex = qpost.create_information_df(analysis, meta_df)

In [ ]:
df = df_quinex
df = df[df["entity"].str.contains("power plant", case=False, na=False)]
df_upload = df.iloc[0:120]
display(df_upload)
#df_upload.to_excel("test_upload2.xlsx", index=False)

In [ ]:
bad = df_quinex[
    df_quinex['quantity_normalized']
    .astype(str)
    .str.contains("10203040506070809010")
]

display(bad)

In [ ]:
df_path = "../data/quinex/df_quinex.parquet"
qpost.save_df(df_quinex, df_path)

### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")